
# Bengali NLP Safety — V25 BanglaBERT Large `is_toxic` OOF

**Protected Kaggle champion: V14 = 0.586**

V20–V24 showed that small post-processing improvements on our 500-row validation set mostly do not
transfer to the hidden Kaggle labels. V25 therefore changes model capacity instead of adding another rule.

### Model

`csebuetnlp/banglabert_large`

- Bengali-specific ELECTRA discriminator
- about 335M parameters
- official BanglaBERT normalization is applied before tokenization
- trained only for `is_toxic`, because `is_toxic` contributes **50%** of the official score

### Training plan

- grouped 5-fold OOF on train + validation
- 3 epochs
- snapshot average of epochs 2 and 3
- max length 128
- T4-safe batch size 2 + gradient accumulation 8
- no gradient checkpointing
- conservative blend against the proven V14 toxic probability
- optional small V17 diversity blend
- fast vectorized threshold search (no slow Cell #10 problem)

V14 remains untouched unless V25 passes the stability gate.

## Run

1. Open this notebook in Google Colab.
2. Select **T4 GPU**.
3. Run cell 1 and upload **`v25_data.zip`**.
4. Run all remaining cells.
5. The model is much larger than our earlier BanglaBERT, so expect this run to take substantially longer.
6. Download **`v25_results.zip`** at the end.
7. Send that ZIP to ChatGPT before submitting any V25 CSV.


In [ ]:

# 1. Upload V25 data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v25_data.zip" in uploaded, "Upload the supplied v25_data.zip."

DATA_DIR = pathlib.Path("/content/v25_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v25_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install dependencies + official BanglaBERT normalizer
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

# Official normalization package recommended by the BanglaBERT authors.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig, AutoTokenizer
from normalizer import normalize

MODEL_NAME = "csebuetnlp/banglabert_large"

print("Testing access to:", MODEL_NAME)
_ = AutoConfig.from_pretrained(MODEL_NAME)
_ = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Normalizer test:", normalize("আমি   ভালো আছি।"))
print("✅ BanglaBERT Large preflight passed.")


In [ ]:

# 3. Imports, settings, data
import gc, re, math, random, json, zipfile, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from normalizer import normalize

SEED = 42
N_FOLDS = 5
MODEL_NAME = "csebuetnlp/banglabert_large"
MAX_LENGTH = 128
EPOCHS = 3
SNAPSHOT_EPOCHS = [2,3]
TRAIN_BATCH_SIZE = 2
ACCUM_STEPS = 8
EVAL_BATCH_SIZE = 8
LR_BACKBONE = 1.0e-5
LR_HEAD = 2.0e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

MIN_GAIN = 0.003
MIN_WINS = 3
MAX_LOSSES = 1

LABELS = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable T4 GPU in Colab."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2v = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
vf = pd.read_csv(DATA_DIR/"validation_folds.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})

with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z:
    old_xlv=z["valid"].copy()
    old_xlt=z["test"].copy()

with np.load(DATA_DIR/"v17_banglabert_oof_test.npz") as z:
    v17_bo=z["oof"].copy()
    v17_bt=z["test"].copy()

with np.load(DATA_DIR/"v17_xlmr_oof_test.npz") as z:
    v17_xo=z["oof"].copy()
    v17_xt=z["test"].copy()

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert champ["ID"].tolist()==test["ID"].tolist()
assert v2v["ID"].tolist()==valid["ID"].tolist()
assert v2t["ID"].tolist()==test["ID"].tolist()

def normalize_for_model(x):
    x = unicodedata.normalize("NFC", str(x))
    x = re.sub(r"\s+", " ", x).strip()
    try:
        x = normalize(x)
    except Exception:
        pass
    return re.sub(r"\s+", " ", x).strip()

def group_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    return re.sub(r"\s+", " ", x).strip().lower()

def has_bengali(x):
    return bool(re.search(r"[\u0980-\u09FF]", str(x)))

combined = pd.concat([train,valid],ignore_index=True)
combined["model_text"] = combined["text"].map(normalize_for_model)
test["model_text"] = test["text"].map(normalize_for_model)
combined["group_text"] = combined["text"].map(group_text)

valid_has_bn = valid["text"].map(has_bengali).to_numpy(bool)
test_has_bn = test["text"].map(has_bengali).to_numpy(bool)

print("combined/test:",len(combined),len(test))
print("toxic positives:",int(combined.is_toxic.sum()))
print("validation no-Bengali-script:",int((~valid_has_bn).sum()))


In [ ]:

# 4. Balanced grouped 5-fold split
gs=combined.groupby("group_text",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

gp=np.array([combined[l].sum() for l in LABELS],float)
total=float(len(combined))
rarity=1/np.sqrt(np.maximum(gp,1))
mat=gs[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+.005*gs["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(gs))*1e-6))

fp=np.zeros((N_FOLDS,len(LABELS)))
fs=np.zeros(N_FOLDS)
assign={}

for idx in order:
    row=gs.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])
    costs=[]
    for f in range(N_FOLDS):
        p=fp.copy(); s=fs.copy()
        p[f]+=vec; s[f]+=size
        costs.append(
            np.mean((p/np.maximum(gp[None,:],1)-1/N_FOLDS)**2)
            + .35*np.mean((s/total-1/N_FOLDS)**2)
        )
    best=int(np.argmin(costs))
    assign[row["group_text"]]=best
    fp[best]+=vec
    fs[best]+=size

combined["fold"]=combined["group_text"].map(assign).astype(int)

counts=combined.groupby("fold").size().reindex(range(5),fill_value=0)
pos=combined.groupby("fold")["is_toxic"].sum().reindex(range(5),fill_value=0)

assert (counts>0).all()
assert (pos>0).all()
assert combined.groupby("group_text")["fold"].nunique().max()==1

print("✅ fold sizes:",counts.to_dict())
print("toxic positives:",pos.to_dict())


In [ ]:

# 5. Dataset helpers
class ToxicDataset(Dataset):
    def __init__(self,frame,tokenizer,with_labels=True):
        self.texts=frame["model_text"].tolist()
        self.targets=frame["is_toxic"].to_numpy(np.float32) if with_labels else None
        self.tokenizer=tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,i):
        enc=self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.targets is not None:
            enc["labels"]=self.targets[i]
        return enc

class Collator:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                [x.pop("labels") for x in items],
                dtype=torch.float32
            )
        batch=self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )
        if y is not None:
            batch["labels"]=y
        return batch

@torch.no_grad()
def predict(model,loader):
    assert len(loader.dataset)>0
    model.eval()
    parts=[]

    for batch in loader:
        batch.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp).logits.squeeze(-1)

        parts.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

    out=np.concatenate(parts)
    assert np.isfinite(out).all()
    return out


In [ ]:

# 6. Train BanglaBERT Large 5-fold OOF
RUN_DIR=Path("/content/v25_output")
RUN_DIR.mkdir(exist_ok=True)

oof=np.zeros(len(combined),np.float32)
test_probs=[]
history=[]
fold_seeds=[42,3407,2026,17,1337]

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

# Token length diagnostic.
probe=tokenizer(
    combined["model_text"].sample(min(1000,len(combined)),random_state=SEED).tolist(),
    truncation=False
)
lens=np.array([len(x) for x in probe["input_ids"]])
print("token p50/p95/p99:",np.percentile(lens,[50,95,99]))
print("fraction >128:",float((lens>MAX_LENGTH).mean()))
del probe,lens
gc.collect()

for fold in range(N_FOLDS):
    tr=combined[combined.fold!=fold].copy()
    va=combined[combined.fold==fold].copy()

    assert set(tr["group_text"]).isdisjoint(set(va["group_text"]))

    cache=RUN_DIR/f"banglabert_large_fold_{fold}.npz"
    hist_path=RUN_DIR/f"banglabert_large_fold_{fold}_history.csv"

    if cache.exists():
        with np.load(cache,allow_pickle=False) as z:
            idx=z["va_index"]; vp=z["valid"]; tp=z["test"]
        if (
            np.array_equal(idx,va.index.to_numpy())
            and len(vp)==len(va)
            and len(tp)==len(test)
        ):
            print("✅ reuse fold",fold)
            oof[idx]=vp
            test_probs.append(tp)
            if hist_path.exists():
                history.extend(pd.read_csv(hist_path).to_dict("records"))
            continue

    print("\n"+"="*70)
    print("fold",fold,"train",len(tr),"valid",len(va))

    gc.collect()
    torch.cuda.empty_cache()
    seed_everything(fold_seeds[fold])

    collator=Collator(tokenizer)

    tl=DataLoader(
        ToxicDataset(tr,tokenizer,True),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator,
    )

    vl=DataLoader(
        ToxicDataset(va,tokenizer,True),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator,
    )

    el=DataLoader(
        ToxicDataset(test,tokenizer,False),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator,
    )

    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
    ).to("cuda")

    # Keep standard backward graph; safer than checkpointing in our Colab stack.
    if hasattr(model,"gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

    pos=float(tr.is_toxic.sum())
    neg=float(len(tr)-pos)
    pos_weight=np.sqrt(neg/max(pos,1))

    loss_fn=torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight],
            dtype=torch.float32,
            device="cuda"
        )
    )

    head_params=[]
    backbone_params=[]

    for n,p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "classifier" in n:
            head_params.append(p)
        else:
            backbone_params.append(p)

    optimizer=torch.optim.AdamW([
        {
            "params":backbone_params,
            "lr":LR_BACKBONE,
            "weight_decay":WEIGHT_DECAY
        },
        {
            "params":head_params,
            "lr":LR_HEAD,
            "weight_decay":WEIGHT_DECAY
        },
    ])

    steps=math.ceil(len(tl)/ACCUM_STEPS)*EPOCHS
    scheduler=get_linear_schedule_with_warmup(
        optimizer,
        int(steps*WARMUP_RATIO),
        steps
    )

    scaler=torch.amp.GradScaler("cuda")
    vs=[]; ts=[]; fh=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running=0.0
        seen=0

        for step,batch in enumerate(
            tqdm(tl,desc=f"BanglaBERT-L f{fold} e{epoch}/{EPOCHS}")
        ):
            y=batch.pop("labels").to("cuda",non_blocking=True)
            inp={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}

            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inp).logits.squeeze(-1)

            loss=loss_fn(logits.float(),y)

            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss")

            scaler.scale(loss/ACCUM_STEPS).backward()

            running+=float(loss.item())*len(y)
            seen+=len(y)

            if (step+1)%ACCUM_STEPS==0 or step+1==len(tl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                old_scale=scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()

                if scaler.get_scale()>=old_scale:
                    scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        rec={
            "fold":fold,
            "epoch":epoch,
            "loss":running/max(seen,1)
        }
        print(rec)
        fh.append(rec)
        pd.DataFrame(fh).to_csv(hist_path,index=False)

        if epoch in SNAPSHOT_EPOCHS:
            vs.append(predict(model,vl))
            ts.append(predict(model,el))

    vp=np.mean(vs,axis=0)
    tp=np.mean(ts,axis=0)

    oof[va.index.to_numpy()]=vp
    test_probs.append(tp)
    history.extend(fh)

    np.savez_compressed(
        cache,
        va_index=va.index.to_numpy(),
        valid=vp,
        test=tp
    )

    del model,tl,vl,el,optimizer,scheduler,scaler,head_params,backbone_params
    gc.collect()
    torch.cuda.empty_cache()

assert len(test_probs)==N_FOLDS

large_test=np.mean(test_probs,axis=0)

np.savez_compressed(
    RUN_DIR/"banglabert_large_oof_test.npz",
    oof=oof,
    test=large_test
)

pd.DataFrame(history).to_csv(
    RUN_DIR/"banglabert_large_training_history.csv",
    index=False
)

print("✅ BanglaBERT Large OOF complete.")


In [ ]:

# 7. Reconstruct V14 toxic probability + prepare validation signals
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

basev=(
    .875*v2v["is_toxic_prob"].to_numpy(float)
    + .125*old_xlv[1:3,:,0].mean(0)
)
baset=(
    .875*v2t["is_toxic_prob"].to_numpy(float)
    + .125*old_xlt[1:3,:,0].mean(0)
)

orig_valid_idx=np.arange(len(train),len(train)+len(valid))
largev=oof[orig_valid_idx]

v17bv=v17_bo[orig_valid_idx]
v17xv=v17_xo[orig_valid_idx]
v17v=.5*v17bv+.5*v17xv
v17t=.5*v17_bt+.5*v17_xt

folds=valid["ID"].map(
    vf.set_index("ID")["fold"]
).to_numpy(int)

y=valid["is_toxic"].to_numpy(int)

assert set(folds)==set(range(5))
print("Prepared.")


In [ ]:

# 8. FAST binary macro-F1 threshold search
def macro_f1(y,pred):
    y=np.asarray(y,dtype=np.int8)
    pred=np.asarray(pred,dtype=np.int8)

    scores=[]
    for cls in [0,1]:
        tp=np.sum((y==cls)&(pred==cls))
        fp=np.sum((y!=cls)&(pred==cls))
        fn=np.sum((y==cls)&(pred!=cls))
        den=2*tp+fp+fn
        scores.append(2*tp/den if den else 0.0)

    return float(np.mean(scores))

def best_threshold(y,p):
    y=np.asarray(y,dtype=np.int8)
    p=np.asarray(p,float)

    order=np.argsort(p)
    ps=p[order]
    ys=y[order]

    prefix_pos=np.r_[0,np.cumsum(ys)]
    prefix_neg=np.r_[0,np.cumsum(1-ys)]

    vals=np.unique(ps)
    if len(vals)>600:
        vals=np.quantile(vals,np.linspace(0,1,600))

    cand=np.unique(np.r_[0.,vals,1.000001])
    idx=np.searchsorted(ps,cand,side="left")

    total_pos=prefix_pos[-1]
    total_neg=prefix_neg[-1]

    tp=total_pos-prefix_pos[idx]
    fp=total_neg-prefix_neg[idx]
    fn=prefix_pos[idx]
    tn=prefix_neg[idx]

    f1p=np.divide(
        2*tp,2*tp+fp+fn,
        out=np.zeros_like(tp,dtype=float),
        where=(2*tp+fp+fn)!=0
    )

    f1n=np.divide(
        2*tn,2*tn+fp+fn,
        out=np.zeros_like(tn,dtype=float),
        where=(2*tn+fp+fn)!=0
    )

    scores=(f1p+f1n)/2
    best=scores.max()

    tied=cand[
        np.isclose(scores,best,atol=1e-12,rtol=0)
    ]

    return float(np.median(tied))

def crossfit(p):
    pred=np.zeros(len(y),dtype=np.int8)
    fs=[]; ts=[]

    for fold in range(5):
        held=folds==fold
        t=best_threshold(y[~held],p[~held])
        ts.append(t)
        pred[held]=(p[held]>=t).astype(np.int8)
        fs.append(macro_f1(y[held],pred[held]))

    return {
        "score":macro_f1(y,pred),
        "fold_scores":np.asarray(fs,float),
        "thresholds":np.asarray(ts,float),
        "prediction":pred,
    }

basecv=crossfit(basev)
largecv=crossfit(largev)

print("V14 toxic:",basecv["score"],np.round(basecv["fold_scores"],4))
print("Pure BanglaBERT Large:",largecv["score"],np.round(largecv["fold_scores"],4))


In [ ]:

# 9. Conservative V25 blend search
rows=[]
best=None
candidates=[]

# V14 + BanglaBERT Large
for mode in ["prob","logit"]:
    for wi in range(1,13):  # .05 ... .60
        w=wi/20

        if mode=="prob":
            p=(1-w)*basev+w*largev
        else:
            p=sigmoid(
                (1-w)*logit(basev)
                + w*logit(largev)
            )

        candidates.append({
            "kind":"base_large",
            "mode":mode,
            "w_large":w,
            "prob":p,
        })

# Script-aware blend.
for mode in ["prob","logit"]:
    for w_bn in [0.05,0.10,0.20,0.30,0.40]:
        for w_non in [0.0,0.05,0.10,0.20,0.30]:
            if abs(w_bn-w_non)>0.30:
                continue

            ww=np.where(valid_has_bn,w_bn,w_non)

            if mode=="prob":
                p=(1-ww)*basev+ww*largev
            else:
                p=sigmoid(
                    (1-ww)*logit(basev)
                    + ww*logit(largev)
                )

            candidates.append({
                "kind":"script_aware",
                "mode":mode,
                "w_bn":w_bn,
                "w_non":w_non,
                "prob":p,
            })

# Small V17 diversity component.
for mode in ["prob","logit"]:
    for wb,wl,w17 in [
        (.80,.15,.05),
        (.75,.20,.05),
        (.75,.15,.10),
        (.70,.25,.05),
        (.70,.20,.10),
        (.65,.25,.10),
    ]:
        if mode=="prob":
            p=wb*basev+wl*largev+w17*v17v
        else:
            p=sigmoid(
                wb*logit(basev)
                + wl*logit(largev)
                + w17*logit(v17v)
            )

        candidates.append({
            "kind":"large_plus_v17",
            "mode":mode,
            "w_base":wb,
            "w_large":wl,
            "w_v17":w17,
            "prob":p,
        })

for c in candidates:
    p=c.pop("prob")
    res=crossfit(p)

    wins=int(
        (res["fold_scores"]>basecv["fold_scores"]+1e-12).sum()
    )
    losses=int(
        (res["fold_scores"]<basecv["fold_scores"]-1e-12).sum()
    )
    ties=5-wins-losses
    gain=res["score"]-basecv["score"]

    passes=(
        gain>=MIN_GAIN
        and wins>=MIN_WINS
        and losses<=MAX_LOSSES
    )

    rec={
        **c,
        "score":res["score"],
        "baseline_score":basecv["score"],
        "gain":gain,
        "wins":wins,
        "ties":ties,
        "losses":losses,
        "passes_gate":passes,
        "median_threshold":float(np.median(res["thresholds"])),
    }

    rows.append(rec)

    if rec["kind"]=="base_large":
        intervention=rec["w_large"]
    elif rec["kind"]=="script_aware":
        intervention=(rec["w_bn"]+rec["w_non"])/2
    else:
        intervention=1-rec["w_base"]

    key=(
        passes,
        res["score"],
        wins,
        -losses,
        -intervention
    )

    if best is None or key>best["key"]:
        best={
            "key":key,
            "rec":rec,
            "cv":res
        }

search=pd.DataFrame(rows).sort_values(
    ["passes_gate","gain","wins","losses"],
    ascending=[False,False,False,True]
)
search.to_csv(
    RUN_DIR/"v25_blend_search.csv",
    index=False
)

display(search.head(30).round(5))
print("BEST:",best["rec"])


In [ ]:

# 10. Build controlled V25 submission
rec=best["rec"]
passes=bool(rec["passes_gate"])

if rec["kind"]=="base_large":
    w=float(rec["w_large"])

    if rec["mode"]=="prob":
        pt=(1-w)*baset+w*large_test
    else:
        pt=sigmoid(
            (1-w)*logit(baset)
            + w*logit(large_test)
        )

elif rec["kind"]=="script_aware":
    ww=np.where(
        test_has_bn,
        float(rec["w_bn"]),
        float(rec["w_non"])
    )

    if rec["mode"]=="prob":
        pt=(1-ww)*baset+ww*large_test
    else:
        pt=sigmoid(
            (1-ww)*logit(baset)
            + ww*logit(large_test)
        )

else:
    wb=float(rec["w_base"])
    wl=float(rec["w_large"])
    w17=float(rec["w_v17"])

    if rec["mode"]=="prob":
        pt=wb*baset+wl*large_test+w17*v17t
    else:
        pt=sigmoid(
            wb*logit(baset)
            + wl*logit(large_test)
            + w17*logit(v17t)
        )

threshold=float(
    np.median(best["cv"]["thresholds"])
)

pred=(pt>=threshold).astype(np.int8)

sub=champ.copy()
before=sub["is_toxic"].to_numpy(int)

if passes:
    sub["is_toxic"]=pred

changed=int(
    (before!=sub["is_toxic"].to_numpy(int)).sum()
)

sub.to_csv(
    RUN_DIR/"submission_v25_toxic.csv",
    index=False
)

report={
    "version":"v25_banglabert_large_toxic",
    "champion_score":0.586,
    "baseline_crossfit":float(basecv["score"]),
    "pure_large_crossfit":float(largecv["score"]),
    "selected":rec,
    "passes_gate":passes,
    "changed_rows":changed,
    "positives_before":int(before.sum()),
    "positives_after":int(sub["is_toxic"].sum()),
    "warning":"Send v25_results.zip to ChatGPT before Kaggle submission."
}

(RUN_DIR/"v25_report.json").write_text(
    json.dumps(report,indent=2)
)

print(report)


In [ ]:

# 11. Package V25 results
zip_path=Path("/content/v25_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
